# Diabetes Dataset Analysis

This notebook performs a reproducible exploratory and statistical analysis of the `diabetes.csv` dataset, covering:

1. **Understand Data Structure**: Inspect dimensions, data types, and missingness.
2. **Clean and Preprocess Data**: Treat clinically implausible zero measurements as missing and apply median imputation for exploratory analysis.
3. **Outlier Review**: Flag potential outliers using the IQR rule without automatically altering clinically plausible observations.
4. **Exploratory Data Analysis (EDA)**: Examine distributions, correlations, and outcome-group patterns.
5. **Statistical Analysis**: Use Welch's t-tests with Benjamini–Hochberg false-discovery-rate correction.
6. **Summarize Findings**: Present cautious, clinically oriented interpretations and limitations.

> **Portfolio note:** This is an exploratory analysis. It is not a diagnostic tool and does not establish causality.


## Understand Data Structure

### Subtask:
Display basic information about the `diabetes.csv` dataset, such as the shape (number of rows and columns), data types of each column, and a summary of missing values. This helps in understanding the overall structure and quality of the data.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

# Reproducible data loading:
# - repository/local execution: ./diabetes.csv
# - Google Colab fallback: /content/diabetes.csv
data_candidates = [Path("diabetes.csv"), Path("/content/diabetes.csv")]
data_path = next((path for path in data_candidates if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "diabetes.csv was not found. Place it in the same folder as this notebook "
        "or upload it to /content/ when using Google Colab."
    )

df_raw = pd.read_csv(data_path)
df = df_raw.copy()

print(f"Loaded dataset from: {data_path.resolve()}")
display(df.head())


In [ ]:
# Display DataFrame shape
print(f"DataFrame shape: {df.shape}")

# Display basic information about the DataFrame
print("\nDataFrame Info:")
df.info()

# Display the count of missing values for each column
print("\nMissing values (initial check):")
print(df.isnull().sum())

## Clean and Preprocess Data

### Subtask:
Perform initial data cleaning steps for the `diabetes.csv` dataset. This involves handling implicit missing values (represented as 0s), correcting inconsistencies, converting data types if necessary, and identifying/addressing outliers.

The initial data contain no explicit `NaN` values, but this dataset uses zero in several physiological measurement columns. In this analysis, zeros in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` are treated as likely missing/unrecorded measurements rather than valid observations.

For exploratory analysis, these zeros are converted to `NaN` and imputed with the column median. The untouched source data remain available in `df_raw` for comparison and reproducibility.


In [ ]:
# Columns where zero is treated as a likely missing/unrecorded measurement
columns_with_zeros_as_nan = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

zero_counts = (df[columns_with_zeros_as_nan] == 0).sum().rename("Zero count before cleaning")
display(zero_counts.to_frame())

# Replace selected zero measurements with NaN
df[columns_with_zeros_as_nan] = df[columns_with_zeros_as_nan].replace(0, np.nan)

print("Missing values after replacing selected zeros with NaN:")
display(df.isna().sum().to_frame("Missing values"))


For this exploratory notebook, missing measurements are imputed using the median of each feature. Median imputation is simple and relatively robust to skewed values.

> If this dataset is later used for predictive modeling, imputation must be fitted **inside the training pipeline** after the train/test split to avoid data leakage.


In [ ]:
imputation_medians = df[columns_with_zeros_as_nan].median()

for column in columns_with_zeros_as_nan:
    df[column] = df[column].fillna(imputation_medians[column])

print("Median values used for imputation:")
display(imputation_medians.to_frame("Imputation median"))

print("Missing values after median imputation:")
display(df.isna().sum().to_frame("Missing values"))


After imputing missing values, it's good practice to check the descriptive statistics again to see the impact of these changes and to identify any potential outliers. I will display the descriptive statistics and then visualize distributions with box plots to identify outliers.

In [ ]:
print("Descriptive statistics after imputation:")
print(df.describe())

To visually identify potential outliers and understand the distribution of each numerical feature, box plots are highly effective. This step will generate box plots for all numerical columns to visualize their distributions and highlight any extreme values that might be considered outliers.

In [ ]:
# Select numerical predictors, excluding the target
numerical_cols = df.select_dtypes(include="number").columns.tolist()
numerical_cols.remove("Outcome")

plt.figure(figsize=(15, 10))

for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i + 1)
    sns.boxplot(y=df[col])
    plt.title(f"Box Plot of {col}")
    plt.ylabel("")

plt.tight_layout()
plt.show()


The box plots show potential extreme observations in several features. In clinical data, an extreme value is not automatically an error, so this notebook **does not cap or delete values solely because they fall outside the IQR range**.

Instead, the IQR rule is used as a transparent diagnostic to count and review potential outliers while preserving the imputed dataset for EDA and statistical testing.


In [ ]:
outlier_records = []

for col in numerical_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    mask = (df[col] < lower_bound) | (df[col] > upper_bound)

    outlier_records.append(
        {
            "Feature": col,
            "Lower IQR bound": lower_bound,
            "Upper IQR bound": upper_bound,
            "Potential outliers": int(mask.sum()),
            "Percent of rows": 100 * mask.mean(),
        }
    )

outlier_summary = pd.DataFrame(outlier_records)
display(outlier_summary.round(2))


## Exploratory Data Analysis (EDA)

### Subtask:
Conduct exploratory data analysis on the `diabetes.csv` dataset to uncover patterns, relationships, and insights. This can involve descriptive statistics, correlation analysis, and initial visualizations (e.g., histograms, scatter plots) for key medical features related to diabetes.

I will calculate the correlation matrix for all numerical features and visualize it using a heatmap to identify linear relationships between variables.

In [ ]:
# Calculate the correlation matrix
corr_matrix = df.corr()

# Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Diabetes Dataset Features')
plt.show()

Next, I will visualize the distribution of the target variable, 'Outcome', using a count plot. This will show the proportion of diabetic and non-diabetic individuals in the dataset. Additionally, I will create histograms for all numerical features to understand their distributions.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='Outcome', data=df, hue='Outcome', palette='viridis', legend=False)
plt.title('Distribution of Diabetes Outcome')
plt.xlabel('Outcome (0: Non-Diabetic, 1: Diabetic)')
plt.ylabel('Count')
plt.show()

# Create histograms for all numerical features
plt.figure(figsize=(20, 15))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i + 1) # Adjust subplot grid as needed
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

To further explore the relationship between each numerical feature and the 'Outcome', I will create violin plots. These plots combine aspects of box plots and kernel density plots, showing both the distribution and central tendencies for each outcome group.

In [ ]:
plt.figure(figsize=(20, 20))
for i, col in enumerate(numerical_cols):
    plt.subplot(4, 2, i + 1) # Adjust subplot grid as needed
    sns.violinplot(x='Outcome', y=col, data=df, hue='Outcome', palette='viridis', legend=False)
    plt.title(f'Violin Plot of {col} by Outcome')
    plt.xlabel('Outcome (0: Non-Diabetic, 1: Diabetic)')
    plt.ylabel(col)

plt.tight_layout()
plt.show()

Finally for the EDA section, I will use a `pairplot` to visualize the pairwise relationships and distributions of all numerical features, colored by the target variable 'Outcome'. This provides a comprehensive overview of how features interact with each other and how they separate the two outcome classes.

In [ ]:
sns.pairplot(df, hue='Outcome', palette='viridis', diag_kind='kde')
plt.suptitle('Pair Plot of Diabetes Dataset Features by Outcome', y=1.02) # Adjust title position
plt.show()

## Visualize Key Features

### Subtask:
Create visualizations to better understand the distribution and relationships of important variables in the `diabetes.csv` dataset. For example, visualize patient demographics, glucose levels, or insulin levels. Ensure all plots have legends for clarity.

I will create a scatter plot of 'Glucose' versus 'Insulin', colored by 'Outcome'. This will help visualize the relationship between these two critical features in diabetes and how they differ between diabetic and non-diabetic individuals.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Glucose', y='Insulin', hue='Outcome', data=df, palette='viridis', s=100, alpha=0.7)
plt.title('Glucose vs. Insulin by Outcome')
plt.xlabel('Glucose')
plt.ylabel('Insulin')
plt.legend(title='Outcome (0: Non-Diabetic, 1: Diabetic)')
plt.grid(True)
plt.show()

Next, I will create a scatter plot of 'BMI' versus 'Age', colored by 'Outcome'. This visualization helps understand the interplay between these two demographic and physiological factors in relation to diabetes diagnosis.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='BMI', y='Age', hue='Outcome', data=df, palette='viridis', s=100, alpha=0.7)
plt.title('BMI vs. Age by Outcome')
plt.xlabel('BMI')
plt.ylabel('Age')
plt.legend(title='Outcome (0: Non-Diabetic, 1: Diabetic)')
plt.grid(True)
plt.show()

To assess if the number of pregnancies is a distinguishing factor, I will create a grouped bar plot showing the average number of pregnancies for diabetic and non-diabetic individuals.

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x='Outcome', y='Pregnancies', data=df, hue='Outcome', palette='viridis', legend=False)
plt.title('Average Pregnancies by Outcome')
plt.xlabel('Outcome (0: Non-Diabetic, 1: Diabetic)')
plt.ylabel('Average Pregnancies')
plt.grid(axis='y')
plt.show()

Finally, I will create a histogram of 'Age' for both diabetic and non-diabetic individuals to visualize and compare their age distributions. This will help understand if certain age groups are more prone to diabetes.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='Age', hue='Outcome', multiple='stack', kde=True, palette='viridis')
plt.title('Distribution of Age by Outcome')
plt.xlabel('Age')
plt.ylabel('Count')
plt.grid(True)
plt.show()

### Comparison of Insulin Data Before and After Imputation

Because `Insulin` contains many zero-coded measurements, the table below documents how many values were treated as missing and the median used for imputation. This makes the preprocessing decision explicit rather than hiding it.


In [ ]:
insulin_zero_mask = df_raw["Insulin"].eq(0)
observed_insulin = df_raw.loc[~insulin_zero_mask, "Insulin"]

insulin_audit = pd.DataFrame(
    {
        "Metric": [
            "Rows in dataset",
            "Zero-coded insulin values",
            "Observed non-zero insulin values",
            "Median of observed insulin",
            "Median used for imputation",
        ],
        "Value": [
            len(df_raw),
            int(insulin_zero_mask.sum()),
            int((~insulin_zero_mask).sum()),
            observed_insulin.median(),
            imputation_medians["Insulin"],
        ],
    }
)

display(insulin_audit)


### Statistical Significance Testing

To compare continuous predictors between `Outcome=0` and `Outcome=1`, this notebook uses **Welch's independent-samples t-test**, which does not require equal population variances.

Because several features are tested, raw p-values are adjusted with the **Benjamini–Hochberg false discovery rate (FDR)** procedure. The table also reports group means and Cohen's *d* as an effect-size estimate.

Statistical significance should be interpreted alongside effect size, data quality, and clinical context.


In [ ]:
# Separate outcome groups
df_non_diabetic = df.loc[df["Outcome"] == 0]
df_diabetic = df.loc[df["Outcome"] == 1]

def cohen_d(group_a, group_b):
    """Standardized mean difference using pooled sample standard deviation."""
    a = np.asarray(group_a, dtype=float)
    b = np.asarray(group_b, dtype=float)

    n_a, n_b = len(a), len(b)
    var_a = a.var(ddof=1)
    var_b = b.var(ddof=1)
    pooled_var = ((n_a - 1) * var_a + (n_b - 1) * var_b) / (n_a + n_b - 2)

    if pooled_var <= 0:
        return np.nan

    return (a.mean() - b.mean()) / np.sqrt(pooled_var)

def benjamini_hochberg(p_values):
    """Benjamini–Hochberg FDR-adjusted p-values."""
    p_values = np.asarray(p_values, dtype=float)
    n_tests = len(p_values)

    order = np.argsort(p_values)
    ranked_p = p_values[order]

    adjusted_ranked = ranked_p * n_tests / np.arange(1, n_tests + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted_ranked = np.clip(adjusted_ranked, 0, 1)

    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = adjusted_ranked
    return adjusted

results = []

for col in numerical_cols:
    group_0 = df_non_diabetic[col].dropna()
    group_1 = df_diabetic[col].dropna()

    t_stat, p_value = stats.ttest_ind(group_0, group_1, equal_var=False)

    results.append(
        {
            "Feature": col,
            "Mean Outcome=0": group_0.mean(),
            "Mean Outcome=1": group_1.mean(),
            "Mean difference (1-0)": group_1.mean() - group_0.mean(),
            "Cohen's d (1-0)": cohen_d(group_1, group_0),
            "Welch t-statistic": t_stat,
            "Raw p-value": p_value,
        }
    )

stats_results = pd.DataFrame(results)
stats_results["FDR-adjusted p-value"] = benjamini_hochberg(stats_results["Raw p-value"])
stats_results["Significant after FDR (0.05)"] = stats_results["FDR-adjusted p-value"] < 0.05

display(
    stats_results.sort_values("FDR-adjusted p-value")
    .reset_index(drop=True)
    .round(4)
)

print(
    "\nInterpretation: FDR-adjusted p < 0.05 indicates evidence of a group-mean "
    "difference after controlling the false discovery rate across these feature tests."
)


## Final Task

### Subtask:
Summarize the main findings from data loading, cleaning, exploratory analysis, and statistical testing, while separating exploratory associations from causal or diagnostic claims.


### Summary of Findings

**Data structure**
- The dataset contains 768 rows and 9 columns, including eight predictors and the binary `Outcome` target.
- Several physiological measurement columns contain zero-coded values that are treated in this notebook as likely missing/unrecorded measurements.

**Cleaning and preprocessing**
- Zero-coded values in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` are converted to missing values and median-imputed for this exploratory analysis.
- The original source data are preserved separately in `df_raw`.
- Potential outliers are flagged with the IQR rule but are **not automatically capped or removed**, because extreme clinical measurements may be genuine observations.

**Exploratory analysis**
- The notebook examines feature distributions, outcome balance, correlations, and outcome-group patterns using heatmaps, histograms, violin plots, pair plots, and targeted clinical-variable visualizations.
- `Glucose` is a particularly important feature to inspect in relation to the diabetes outcome, while BMI, age, pregnancies, insulin, and diabetes pedigree information also warrant interpretation in combination rather than isolation.

**Statistical analysis**
- Outcome groups are compared using Welch's independent-samples t-tests.
- Benjamini–Hochberg FDR correction is applied across the feature tests.
- Effect sizes are reported alongside p-values so that statistical evidence is not interpreted without magnitude.

### Limitations and Next Steps

1. **Do not infer causality** from this observational, exploratory analysis.
2. **Imputation uncertainty** is not captured by simple single-median imputation.
3. **Outlier review** should use source documentation and clinical context before any value is altered.
4. For machine learning, preprocessing must be fitted inside cross-validation/training pipelines to avoid leakage.
5. Future modeling should report clinically useful metrics such as sensitivity/recall, specificity, precision, F1, ROC-AUC/PR-AUC, and calibration—not accuracy alone.
6. Any predictive use would require external validation before clinical application.
